# Bonus Slideshow Vidéo - Générateur de Slideshow (Optionnel)

**Navigation** : [<< Précédent (01-1)](01-1-Video-Operations-Basics.ipynb) | [Index](../../README.md) | [Suivant >> (01-2)](01-2-GPT-5-Video-Understanding.ipynb)

**Module :** 01-Video-Foundation  
**Niveau :** Debutant  
**Type :** Accrétion bonus (optionnelle)  
**Duree estimee :** 15 minutes  
**VRAM :** 0 (CPU uniquement)

> **Ce notebook est une accrétion optionnelle**, extraite de la conclusion de
> [01-1-Video-Opérations-Basics](01-1-Video-Operations-Basics.ipynb) pour rendre la
> lane principale plus claire. Il reprend intégralement le challenge qui figurait en
> fin du 01-1, sans modification du fond ni de la forme. Les étudiants qui veulent
> transformer les manipulations moviepy/imageio du 01-1 en **produit livrable**
> trouvent ici le terrain de jeu ; les autres passent directement au 01-2.

## Contexte pédagogique

Le 01-1 a posé les fondamentaux :

- **Section 1** : création d'une vidéo de test à partir de frames générées avec PIL
  (`Image.new`, `ImageDraw`) et assemblage avec `imageio.get_writer().append_data()`.
- **Sections 2-4** : manipulations moviepy (trim, concaténation, overlay texte),
  sonde ffmpeg-python, extraction rapide de frames avec decord.

Le challenge ci-dessous remet ces primitives au service d'un **livrable fini** : un
slideshow 5 secondes @ 1 FPS avec cinq messages sur cinq fonds colorés différents.
C'est la transition entre « savoir manipuler une vidéo » et « savoir produire un
objet vidéo autonome ».

## Prérequis

- Avoir complété (ou parcouru) le notebook 01-1.
- Packages : `Pillow`, `imageio`, `imageio-ffmpeg` (déjà listés dans
  `requirements-video.txt` à la racine de `GenAI/`).
- FFmpeg installé sur le système (`winget install FFmpeg` ou `apt install ffmpeg`).


***

## CHALLENGE BONUS - Générateur de Slideshow Vidéo

**Points : 0.5 pts**

## Objectif

Créer une vidéo **slideshow** à partir de frames générées programmatiquement.

## Ce que vous avez appris

La Section 1 montre:
- Génération de frames avec PIL (`Image.new`, `ImageDraw`)
- Assemblage avec imageio (`get_writer`, `append_data`)

## Critères de succes

- [ ] Générer **5 frames** affichant un message différent chacune:
  - Frame 1: "Bienvenue"
  - Frame 2: "Voici le sujet"
  - Frame 3: "Point important"
  - Frame 4: "Conclusion"
  - Frame 5: "Merci"
- [ ] Chaque frame doit avoir:
  - Une couleur de fond différente (gradient ou palette cohérente)
  - Le texte centré
- [ ] Assembler en vidéo **5 secondes @ 1 FPS**
- [ ] Sauvegarder dans `OUTPUT_DIR`

## Contraintes techniques

- Résolution: 640x480
- Utiliser PIL pour les frames, imageio pour l'assemblage
- FPS = 1 signifie chaque frame dure 1 seconde

***

**Soumission** : PR avec titre "Challenge #7 - [Votre Nom]", aperçu des frames et vidéo générée

## Espace de travail du challenge

Le challenge ci-dessus était resté **sans endroit où répondre** : ce notebook ajoute l'espace de travail. Deux **exemples guidés** rappellent les primitives posées dans [01-1](01-1-Video-Operations-Basics.ipynb) (une frame PIL centrée, un mini assemblage imageio) ; trois **exercices** portent le livrable complet. Tout tourne sur CPU (VRAM 0, conformément à l'en-tête) : `Pillow` pour les frames, `imageio` + `imageio-ffmpeg` (binaire ffmpeg embarqué) pour l'assemblage. Les artefacts sont écrits dans `outputs/video_01_1b_slideshow/` (non versionné).


### Exemple guidé 1 — Une frame PIL : fond coloré, message centré

La brique de base du slideshow. La fonction `frame_demo` crée une image RGB 640x480, remplit le fond, puis centre le texte en calculant sa boîte englobante :


In [5]:
from pathlib import Path
from PIL import Image, ImageDraw

OUTPUT_DIR = Path("outputs") / "video_01_1b_slideshow"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def frame_demo(message, couleur, taille=(640, 480), chemin=None):
    """Genere une frame PIL : fond colore uniforme, message centre, texte blanc."""
    img = Image.new("RGB", taille, couleur)
    dessin = ImageDraw.Draw(img)
    bbox = dessin.textbbox((0, 0), message)
    x = (taille[0] - (bbox[2] - bbox[0])) // 2
    y = (taille[1] - (bbox[3] - bbox[1])) // 2
    dessin.text((x, y), message, fill="white")
    if chemin is not None:
        img.save(chemin)
    return img


frame_exemple = frame_demo("Demo A", (40, 60, 120), chemin=OUTPUT_DIR / "frame_demo.png")
print(f"Frame exemple sauvegardee : {OUTPUT_DIR / 'frame_demo.png'} ({frame_exemple.size})")


Frame exemple sauvegardee : outputs/video_01_1b_slideshow/frame_demo.png ((640, 480))


### Exemple guidé 2 — Mini assemblage : 2 frames @ 1 FPS

Le montage minimal : à 1 FPS, chaque frame dure une seconde. Deux frames donnent donc une vidéo de 2 secondes — le livrable du challenge est la même mécanique répétée 5 fois :


In [7]:
import numpy as np
import imageio.v2 as imageio

frames_exemple = [
    frame_demo("Demo A", (40, 60, 120)),
    frame_demo("Demo B", (120, 60, 40)),
]
video_exemple = OUTPUT_DIR / "mini_slideshow_exemple.mp4"
with imageio.get_writer(video_exemple, fps=1) as writer:
    for f in frames_exemple:
        writer.append_data(np.asarray(f))
print(f"Video exemple : {video_exemple.name} ({video_exemple.stat().st_size} octets, "
      f"{len(frames_exemple)} frames @ 1 FPS = {len(frames_exemple)} s)")


Video exemple : mini_slideshow_exemple.mp4 (2340 octets, 2 frames @ 1 FPS = 2 s)


## Exercice 1 — Les 5 frames du challenge

Produire la liste des 5 frames imposées par les critères de succès : une par message, chacune sur un fond **distinct** (palette cohérente de votre choix), texte centré, résolution 640x480. Réutiliser `frame_demo` en variant la couleur, ou généraliser si vous préférez un dégradé.


In [9]:
from PIL import ImageFont

MESSAGES = ["Bienvenue", "Voici le sujet", "Point important", "Conclusion", "Merci"]
PALETTE = [(75, 22, 55), (105, 30, 65), (135, 38, 75), (165, 46, 85), (195, 54, 95)]

def creer_frame(message, couleur, taille=(640, 480)):
    """Centre la boite reelle du texte, y compris son decalage vertical."""
    if not message.strip():
        raise ValueError("Le message doit contenir du texte")
    image = Image.new("RGB", taille, couleur)
    dessin = ImageDraw.Draw(image)
    try:
        police = ImageFont.truetype("DejaVuSans.ttf", 36)
    except OSError:
        police = ImageFont.load_default()
    gauche, haut, droite, bas = dessin.textbbox((0, 0), message, font=police)
    position = ((taille[0] - (droite-gauche))/2 - gauche,
                (taille[1] - (bas-haut))/2 - haut)
    dessin.text(position, message, font=police, fill="white")
    return image

frames_challenge = [creer_frame(m, c) for m, c in zip(MESSAGES, PALETTE)]
assert len(frames_challenge) == 5 and len(set(PALETTE)) == 5
assert all(f.size == (640, 480) and f.mode == "RGB" for f in frames_challenge)
for i, frame in enumerate(frames_challenge, 1):
    frame.save(OUTPUT_DIR / f"frame_{i}.png")
try:
    creer_frame("", PALETTE[0])
except ValueError:
    print("Cas limite : message vide refuse")
else:
    raise AssertionError("Un message vide devrait etre refuse")
print("5 images 640x480 generees, couleurs distinctes et texte centre")


Cas limite : message vide refuse
5 images 640x480 generees, couleurs distinctes et texte centre


## Exercice 2 — L'assemblage : 5 secondes @ 1 FPS

Assembler `frames_challenge` en vidéo **slideshow 5 s @ 1 FPS** (5 frames, une seconde chacune), exactement comme la vidéo exemple, et sauvegarder dans `OUTPUT_DIR` sous le nom `slideshow_5s.mp4`.


In [11]:
video_challenge = OUTPUT_DIR / "slideshow_5s.mp4"
with imageio.get_writer(video_challenge, fps=1, codec="libx264", pixelformat="yuv420p") as writer:
    for frame in frames_challenge:
        writer.append_data(np.asarray(frame))
print("Video creee :", video_challenge, video_challenge.stat().st_size, "octets")


Video creee : outputs/video_01_1b_slideshow/slideshow_5s.mp4 13797 octets


## Exercice 3 — Vérifier le livrable

Un livrable se vérifie avant de se soumettre. Contrôler que `slideshow_5s.mp4` existe, que sa taille est non nulle, et que ses métadonnées coïrent avec l'attendu : 5 frames, durée 5 secondes. Indices : `imageio.get_reader(...)` expose `count_frames()` et `get_meta_data()` (clés `fps`, `duration`).


In [13]:
def verifier_video(chemin):
    verdict = {"existe": chemin.is_file(), "nframes": None, "duree_s": None, "ok": False}
    if not verdict["existe"] or chemin.stat().st_size == 0:
        return verdict
    try:
        with imageio.get_reader(chemin) as lecteur:
            meta = lecteur.get_meta_data()
            verdict["nframes"] = lecteur.count_frames()
            verdict["duree_s"] = meta.get("duration")
            verdict["ok"] = (verdict["nframes"] == 5
                and verdict["duree_s"] is not None
                and abs(verdict["duree_s"] - 5.0) < 0.05
                and abs(meta.get("fps", 0) - 1.0) < 0.001
                and tuple(meta.get("size", ())) == (640, 480))
    except (OSError, ValueError, RuntimeError) as erreur:
        verdict["erreur"] = str(erreur)
    return verdict

verdict_livrable = verifier_video(video_challenge)
assert verdict_livrable["ok"], verdict_livrable
assert not verifier_video(OUTPUT_DIR / "absent_test.mp4")["ok"]
print(verdict_livrable)


{'existe': True, 'nframes': 5, 'duree_s': 5.0, 'ok': True}


### Explication de la correction (G15)
Chaque image a une couleur distincte. Le texte est centré à partir de sa boîte englobante, y compris son décalage. Cinq images à une image par seconde produisent une vidéo de cinq secondes. La vérification relit les métadonnées et compte les images ; elle teste aussi un fichier absent. Un message vide est refusé.
Correction préparée avec assistance de ChatGPT/Codex, déclarée.